# NumCompute Stream Demo

This notebook demonstrates chunk-wise streaming learning using only NumPy and matplotlib.


In [ ]:
import sys
sys.path.append('..')

from numcompute_stream import (
    load_csv, train_test_split, make_stream_chunks,
    StreamingImputer, StreamingStandardScaler,
    DecisionTreeClassifier, RandomForestClassifier,
    StreamingPipeline, accuracy_score
)
from numcompute_stream.visualise import plot_model_comparison, plot_metric_history


## 1. Load data using custom io.py


In [ ]:
X, y = load_csv('sample_data.csv', target_column=-1, skip_header=True)
print('X shape:', X.shape)
print('y shape:', y.shape)


## 2. Split into train and test data


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=7)
print(X_train.shape, X_test.shape)


## 3. Create streaming pipelines


In [ ]:
single_tree = StreamingPipeline([
    ('imputer', StreamingImputer()),
    ('scaler', StreamingStandardScaler()),
    ('model', DecisionTreeClassifier(max_depth=5, random_state=1))
])

forest = StreamingPipeline([
    ('imputer', StreamingImputer()),
    ('scaler', StreamingStandardScaler()),
    ('model', RandomForestClassifier(n_estimators=7, max_depth=5, random_state=1))
])


## 4. Train incrementally using partial_fit()


In [ ]:
tree_scores = []
forest_scores = []

for chunk_id, (X_chunk, y_chunk) in enumerate(make_stream_chunks(X_train, y_train, chunk_size=20), start=1):
    single_tree.partial_fit(X_chunk, y_chunk)
    forest.partial_fit(X_chunk, y_chunk)

    tree_pred = single_tree.predict(X_test)
    forest_pred = forest.predict(X_test)

    tree_acc = accuracy_score(y_test, tree_pred)
    forest_acc = accuracy_score(y_test, forest_pred)

    tree_scores.append(tree_acc)
    forest_scores.append(forest_acc)

    print(f'Chunk {chunk_id}: tree={tree_acc:.3f}, forest={forest_acc:.3f}')


## 5. Visualise streaming model comparison


In [ ]:
fig, ax = plot_model_comparison(tree_scores, forest_scores)


## 6. Reflection

The single decision tree is faster and easier to interpret, while the random forest is usually more stable because it combines multiple bootstrapped trees. The partial_fit interface allows both models to be updated as new chunks arrive.
